# One-Shot Prompting — CodeLlama-7B-Instruct

This notebook evaluates `codellama/CodeLlama-7b-Instruct-hf` on the
`code_x_glue_cc_defect_detection` test set using **one-shot prompting**
(one labeled example provided before the query).

The model is loaded with 4-bit quantization to fit within Colab T4 VRAM.

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q scikit-learn matplotlib seaborn

In [ ]:
import os
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from utils.data_loader import load_defect_dataset, format_prompt_codellama, get_few_shot_examples
from utils.evaluation import evaluate_predictions, parse_model_output

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Model (4-bit)

In [ ]:
MODEL_ID = "codellama/CodeLlama-7b-Instruct-hf"
MAX_SEQ_LENGTH = 1024
RESULTS_DIR = "../../docs/codellama_docs"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model.eval()
print(f"Model loaded. Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

## 3. Load Dataset and Prepare One-Shot Example

In [ ]:
train_data, _, test_data = load_defect_dataset()
print(f"Test samples: {len(test_data)}")

# Get 1 example (balanced - we pick 1 defective sample)
one_shot_examples = get_few_shot_examples(train_data, n=1, seed=42)
print(f"\nOne-shot example:")
print(f"  Label: {one_shot_examples[0][1]}")
print(f"  Code preview: {one_shot_examples[0][0][:150]}...")

## 4. One-Shot Prompting

Each prompt includes one labeled example before the actual query.
This gives the model context about the expected output format.

In [ ]:
# Preview a sample prompt
sample_prompt = format_prompt_codellama(test_data[0]["func"], examples=one_shot_examples)
print(sample_prompt[:800])
print("...")

In [ ]:
y_true = []
y_pred = []
failed_parses = []

for i, sample in enumerate(tqdm(test_data, desc="One-shot inference")):
    prompt = format_prompt_codellama(sample["func"], examples=one_shot_examples)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = parse_model_output(generated)

    if pred == -1:
        failed_parses.append({"index": i, "output": generated, "label": sample["target"]})
        pred = 0

    y_true.append(sample["target"])
    y_pred.append(pred)

print(f"\nFailed to parse: {len(failed_parses)} / {len(test_data)}")
if failed_parses[:5]:
    print("Sample failed outputs:")
    for fp in failed_parses[:5]:
        print(f"  idx={fp['index']}, output='{fp['output']}'")

## 5. Evaluation

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

metrics = evaluate_predictions(
    y_true, y_pred,
    model_name="codellama",
    strategy="one_shot",
    save_dir=RESULTS_DIR,
)

## 6. Error Analysis

In [ ]:
errors = [
    {"index": i, "true": yt, "pred": yp, "code": test_data[i]["func"][:200]}
    for i, (yt, yp) in enumerate(zip(y_true, y_pred))
    if yt != yp
]

false_positives = [e for e in errors if e["true"] == 0 and e["pred"] == 1]
false_negatives = [e for e in errors if e["true"] == 1 and e["pred"] == 0]

print(f"Total errors: {len(errors)}")
print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")

print("\n--- Sample False Negatives (missed bugs) ---")
for e in false_negatives[:3]:
    print(f"\nIndex: {e['index']}")
    print(f"Code: {e['code']}...")

print("\n--- Sample False Positives (false alarms) ---")
for e in false_positives[:3]:
    print(f"\nIndex: {e['index']}")
    print(f"Code: {e['code']}...")

## 7. Prompt Used

The one-shot prompt template for reference (to submit as best prompting strategy):

In [ ]:
print("=" * 60)
print("ONE-SHOT PROMPT TEMPLATE")
print("=" * 60)
print(format_prompt_codellama("<CODE_HERE>", examples=one_shot_examples))